In [33]:
import numpy as np
import pandas as pd

1. Memuat Dataset

In [34]:
df = pd.read_csv("./src/data/Amazon Product Reviews.csv")

In [35]:
print("Ukuran dataset awal:", df.shape)
print("\nLima baris pertama:")
display(df.head())

Ukuran dataset awal: (568454, 10)

Lima baris pertama:


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


2. Pembersihan & Penanganan Missing Values
- Memeriksa jumlah nilai null pada setiap kolom

In [36]:
print("\nJumlah Missing Value per Kolom:")
print(df.isnull().sum())


Jumlah Missing Value per Kolom:
Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64


- Menangani missing value pada teks/nama profil agar tidak error saat diolah

In [37]:
df["ProfileName"] = df["ProfileName"].fillna("Unknown")
df["Summary"] = df["Summary"].fillna("-")

- Cek kembali nilai setelah penanganan missing value

In [38]:
print("\nJumlah Missing Value per Kolom:")
print(df.isnull().sum())


Jumlah Missing Value per Kolom:
Id                        0
ProductId                 0
UserId                    0
ProfileName               0
HelpfulnessNumerator      0
HelpfulnessDenominator    0
Score                     0
Time                      0
Summary                   0
Text                      0
dtype: int64


3. Konversi Format Waktu (Unix Timestamp ke Datetime)
- Kolom 'Time' berada dalam format Unix Timestamp, dikonversi ke ISO Datetime

In [39]:
df["datetime"] = pd.to_datetime(df["Time"], unit="s")
df["year"] = df["datetime"].dt.year
df["month"] = df["datetime"].dt.month
df["date"] = df["datetime"].dt.date
df["day_of_week"] = df["datetime"].dt.day_name()

4. Penanganan Data Duplikat
- Menghapus ulasan ganda dari UserId yang sama untuk ProductId yang sama pada waktu yang sama

In [40]:
initial_count = len(df)
df = df.drop_duplicates(
    subset=["UserId", "ProductId", "Time", "Text"], keep="first"
)
print(f"\nJumlah data terhapus karena duplikat: {initial_count - len(df)}")


Jumlah data terhapus karena duplikat: 1205


5. Pembersihan Anomali Data Kebermanfaatan (Helpfulness)
- Menghapus baris abnormal di mana HelpfulnessNumerator > HelpfulnessDenominator

In [41]:
df = df[df["HelpfulnessNumerator"] <= df["HelpfulnessDenominator"]]

- Menghitung Rasio Kebermanfaatan (Helpfulness Ratio)

In [42]:
df["HelpfulnessRatio"] = np.where(
    df["HelpfulnessDenominator"] > 0,
    df["HelpfulnessNumerator"] / df["HelpfulnessDenominator"],
    0.0,
)

6. Pemfilteran Data untuk Graph Mining </br>
Graph dengan ~500rb baris bisa sangat lambat saat Community Detection (Louvain).

- Filter reviewer dan produk yang memiliki batas minimum interaksi.

In [43]:
min_user_reviews = 3
min_product_reviews = 5

In [44]:
user_counts = df["UserId"].value_counts()
product_counts = df["ProductId"].value_counts()

active_users = user_counts[user_counts >= min_user_reviews].index
active_products = product_counts[product_counts >= min_product_reviews].index

df_filtered = df[
    df["UserId"].isin(active_users) & df["ProductId"].isin(active_products)
].copy()

In [45]:
print(f"Ukuran dataset setelah diproses: {df.shape}")
print(f"Ukuran dataset terfilter untuk Graf: {df_filtered.shape}")

Ukuran dataset setelah diproses: (567247, 16)
Ukuran dataset terfilter untuk Graf: (290898, 16)


7. Menyimpan Edge List untuk Bipartite Graph
- Node 1: UserId, Node 2: ProductId, Atribut: Score, Time/datetime

In [46]:
edge_list = df_filtered[["UserId", "ProductId", "Score", "Time", "datetime"]]

8. Selesai
- Simpan hasil pembersihan ke file CSV baru

In [47]:
df.to_csv("./src/data/clean_reviews.csv", index=False)
edge_list.to_csv("./src/data/bipartite_edges.csv", index=False)

In [48]:
print("\nPreprocessing selesai! File 'clean_reviews.csv' dan 'bipartite_edges.csv' berhasil disimpan.")


Preprocessing selesai! File 'clean_reviews.csv' dan 'bipartite_edges.csv' berhasil disimpan.
